# Lab: Spark Operations

**Course 1, Week 2: Spark Fundamentals**

## Objectives
- Use select() to choose and transform columns
 - Use filter() to select rows by condition
 - Use groupBy() with aggregation functions
 - Perform joins between DataFrames
 - Understand lazy evaluation vs actions


## Spark Core Concepts

**Key terminology:**
- **SparkSession:** Entry point to Spark functionality (`spark`)
- **DataFrame:** Distributed collection of rows with named columns
- **Transformation:** Lazy operation that defines a computation (select, filter, groupBy)
- **Action:** Triggers computation and returns results (show, count, collect)
- **Catalyst Optimizer:** Spark's query optimizer that plans execution

**Lazy evaluation:** Transformations are not executed until an action is called.
 This allows Spark to optimize the entire query plan.


## Setup: Create Lab Data

In [0]:
from pyspark.sql import functions as F

# Sales data
sales = spark.createDataFrame(
    [
        ("2024-01-01", "S001", "Electronics", "Laptop", 999.99, 2, "West"),
        ("2024-01-01", "S002", "Books", "Python Guide", 49.99, 5, "East"),
        ("2024-01-02", "S003", "Electronics", "Phone", 699.99, 3, "West"),
        ("2024-01-02", "S004", "Clothing", "Jacket", 129.99, 4, "North"),
        ("2024-01-03", "S005", "Books", "Data Science", 59.99, 8, "East"),
        ("2024-01-03", "S006", "Electronics", "Tablet", 449.99, 2, "South"),
        ("2024-01-04", "S007", "Clothing", "Shoes", 89.99, 6, "West"),
        ("2024-01-04", "S008", "Electronics", "Earbuds", 79.99, 10, "North"),
        ("2024-01-05", "S009", "Books", "ML Handbook", 69.99, 3, "South"),
        ("2024-01-05", "S010", "Clothing", "Hat", 29.99, 15, "East"),
    ],
    ["date", "sale_id", "category", "product", "price", "quantity", "region"],
)

# Region lookup
regions = spark.createDataFrame(
    [
        ("West", "Pacific", "Sarah"),
        ("East", "Atlantic", "Mike"),
        ("North", "Central", "Lisa"),
        ("South", "Gulf", "Tom"),
    ],
    ["region", "territory", "manager"],
)

print(f"Sales: {sales.count()} rows")
print(f"Regions: {regions.count()} rows")

Sales: 10 rows
Regions: 4 rows


## Part 1: Select Operations

EXERCISE: Use select() to create derived columns.

In [0]:
# EXERCISE 1a: Select sale_id, product, price, quantity
# YOUR CODE HERE

sales.select('sale_id','product','price','quantity').display()

sale_id,product,price,quantity
S001,Laptop,999.99,2
S002,Python Guide,49.99,5
S003,Phone,699.99,3
S004,Jacket,129.99,4
S005,Data Science,59.99,8
S006,Tablet,449.99,2
S007,Shoes,89.99,6
S008,Earbuds,79.99,10
S009,ML Handbook,69.99,3
S010,Hat,29.99,15


In [0]:

# EXERCISE 1b: Create a total_revenue column (price * quantity)
# and a discounted_price column (price * 0.9)
# YOUR CODE HERE

from pyspark.sql.functions import col

sales.select(
    col("sale_id"),
    col("product"),
    (col("price")*col("quantity")). alias("total_revenue"),
    (col("price")*0.9).alias("discounted_price")
).display()

sale_id,product,total_revenue,discounted_price
S001,Laptop,1999.98,899.991
S002,Python Guide,249.95000000000002,44.991
S003,Phone,2099.9700000000003,629.991
S004,Jacket,519.96,116.99100000000001
S005,Data Science,479.92,53.991
S006,Tablet,899.98,404.99100000000004
S007,Shoes,539.9399999999999,80.991
S008,Earbuds,799.9,71.991
S009,ML Handbook,209.96999999999997,62.991
S010,Hat,449.84999999999997,26.991


## Part 2: Filter Operations

EXERCISE: Use filter() to find specific rows.

In [0]:
# EXERCISE 2a: Filter sales where price > 100
# YOUR CODE HERE

sales.filter(col("price")>100).display()

date,sale_id,category,product,price,quantity,region
2024-01-01,S001,Electronics,Laptop,999.99,2,West
2024-01-02,S003,Electronics,Phone,699.99,3,West
2024-01-02,S004,Clothing,Jacket,129.99,4,North
2024-01-03,S006,Electronics,Tablet,449.99,2,South


In [0]:
# EXERCISE 2b: Filter Electronics sales in the West region
# Hint: Use & for AND conditions
# YOUR CODE HERE

sales.filter((F.col("category")=="Electronics") & (F.col("region")=="West")).display()

date,sale_id,category,product,price,quantity,region
2024-01-01,S001,Electronics,Laptop,999.99,2,West
2024-01-02,S003,Electronics,Phone,699.99,3,West


In [0]:
# EXERCISE 2c: Filter sales between Jan 2 and Jan 4 (inclusive)
# Hint: Use .where() with BETWEEN
# YOUR CODE HERE

sales.where(F.col("date").between("2024-01-02","2024-01-04")).display()

date,sale_id,category,product,price,quantity,region
2024-01-02,S003,Electronics,Phone,699.99,3,West
2024-01-02,S004,Clothing,Jacket,129.99,4,North
2024-01-03,S005,Books,Data Science,59.99,8,East
2024-01-03,S006,Electronics,Tablet,449.99,2,South
2024-01-04,S007,Clothing,Shoes,89.99,6,West
2024-01-04,S008,Electronics,Earbuds,79.99,10,North


## Part 3: GroupBy & Aggregations

EXERCISE: Compute summary statistics.

In [0]:
# EXERCISE 3a: Total revenue by category
# Columns: category, total_revenue (sum of price*quantity), num_sales (count)
# Order by total_revenue descending
# YOUR CODE HERE

sales.groupBy(F.col("category")).agg(
    F.col("category"),
    F.sum(F.col("price")*col("quantity")).alias("total_revenue"),
    F.count("*").alias("num_sales")
).orderBy(F.col("total_revenue"), ascending=False).display()

category,category,total_revenue,num_sales
Electronics,Electronics,5799.83,4
Clothing,Clothing,1509.75,3
Books,Books,939.8399999999999,3


In [0]:
# EXERCISE 3b: Average price and total quantity by region
# YOUR CODE HERE

sales.groupBy("region").agg(
    F.avg(col("price")).alias("avg_price"),
    F.sum(col("quantity")).alias("total_quantity")
).display()

region,avg_price,total_quantity
West,596.6566666666666,11
East,46.656666666666666,28
North,104.99000000000001,14
South,259.99,5


In [0]:
# EXERCISE 3c: Find the most expensive product in each category
# Hint: groupBy("category").agg(F.max("price"))
# YOUR CODE HERE

sales.groupBy("category").agg(
    F.max_by("product", "price").alias("most_expensive_product"),
    F.max("price").alias("max_price")
).display()

category,most_expensive_product,max_price
Electronics,Laptop,999.99
Books,ML Handbook,69.99
Clothing,Jacket,129.99


## Part 4: Joins

EXERCISE: Combine sales with region information.

In [0]:
# EXERCISE 4a: Inner join sales with regions on the region column
# Show: sale_id, product, price, region, territory, manager
# YOUR CODE HERE

sales.join(regions,"region").select(
    col("sale_id"),
    col("product"),
    col("price"),
    col("region"),
    col("territory"),
    col("manager")
).display()

sale_id,product,price,region,territory,manager
S001,Laptop,999.99,West,Pacific,Sarah
S002,Python Guide,49.99,East,Atlantic,Mike
S003,Phone,699.99,West,Pacific,Sarah
S004,Jacket,129.99,North,Central,Lisa
S005,Data Science,59.99,East,Atlantic,Mike
S006,Tablet,449.99,South,Gulf,Tom
S007,Shoes,89.99,West,Pacific,Sarah
S008,Earbuds,79.99,North,Central,Lisa
S009,ML Handbook,69.99,South,Gulf,Tom
S010,Hat,29.99,East,Atlantic,Mike


In [0]:

# EXERCISE 4b: Find total revenue by territory and manager
# (Join first, then groupBy territory and manager)
# YOUR CODE HERE

sales.join(regions,"region").groupBy("territory","manager").agg(
    F.sum(col("price")*col("quantity")).alias("total_revenue")
).display()

territory,manager,total_revenue
Pacific,Sarah,4639.89
Atlantic,Mike,1179.72
Central,Lisa,1319.8600000000001
Gulf,Tom,1109.95


## Part 5: SQL Equivalent

EXERCISE: Register as views and write SQL queries.

In [0]:
sales.createOrReplaceTempView("sales")
regions.createOrReplaceTempView("regions")

In [0]:
%sql
-- EXERCISE: Write a SQL query that finds the top 3 products by total revenue
-- (revenue = price * quantity)
-- YOUR CODE HERE


SELECT 
    product,
    SUM(price*quantity) AS total_revenue
FROM sales
GROUP BY product
ORDER BY total_revenue DESC
LIMIT 3

product,total_revenue
Phone,2099.9700000000003
Laptop,1999.98
Tablet,899.98


## Validation

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Sales data loaded
    checks.append(("Sales data loaded", sales.count() == 10))

    # Check 2: Can perform select
    try:
        result = sales.select("sale_id", "product", "price", "quantity")
        checks.append(("Select operation", len(result.columns) == 4))
    except Exception:
        checks.append(("Select operation", False))

    # Check 3: Can perform filter
    try:
        result = sales.filter(F.col("price") > 100)
        checks.append(("Filter operation", result.count() > 0))
    except Exception:
        checks.append(("Filter operation", False))

    # Check 4: Can perform groupBy
    try:
        result = sales.groupBy("category").agg(F.count("*").alias("n"))
        checks.append(("GroupBy operation", result.count() == 3))
    except Exception:
        checks.append(("GroupBy operation", False))

    # Check 5: Can perform join
    try:
        result = sales.join(regions, "region", "inner")
        checks.append(("Join operation", result.count() == 10))
    except Exception:
        checks.append(("Join operation", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

Lab Validation Results:
----------------------------------------
  [PASS] Sales data loaded
  [PASS] Select operation
  [PASS] Filter operation
  [PASS] GroupBy operation
  [PASS] Join operation

All checks passed! Lab complete.
